In [7]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, max_error
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from config import DATA_DIR

In [8]:
df_parkings_availabilities = pd.read_parquet(DATA_DIR/'parking_availabilities.parquet')

In [9]:
print(df_parkings_availabilities.columns)

Index(['id', 'parking_id', 'spaces_left', 'trend', 'measured_at', 'created_at',
       'updated_at'],
      dtype='object')


In [11]:
FREE_MONTHS = {2, 7, 8, 9}                       # Feb, Jul, Aug, Sep
BUCKET_COLS = ['parking_id', 'is_free', 'day_of_week', 'hour']

def add_bucket_features(df):
    df = df.copy()
    df['measured_at'] = pd.to_datetime(df['measured_at'])
    df['is_free']     = df['measured_at'].dt.month.isin(FREE_MONTHS).astype(int)
    df['day_of_week'] = df['measured_at'].dt.dayofweek           # 0 = Monday
    df['hour']        = df['measured_at'].dt.hour
    return df

# prepare data
df = add_bucket_features(df_parkings_availabilities)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# "fit": mean of spaces_left per bucket
bucket_means = (
    train_df.groupby(BUCKET_COLS)['spaces_left']
            .mean()
            .rename('predicted')
            .reset_index()
)

# fallbacks for buckets unseen in training
parking_means = train_df.groupby('parking_id')['spaces_left'].mean()
global_mean   = train_df['spaces_left'].mean()

# "predict" on test 
preds = test_df.merge(bucket_means, on=BUCKET_COLS, how='left')
preds['predicted'] = preds['predicted'].fillna(preds['parking_id'].map(parking_means))
preds['predicted'] = preds['predicted'].fillna(global_mean)

y_true = preds['spaces_left'].values
y_pred = preds['predicted'].values

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)
max_err = max_error(y_true, y_pred)
print(f"Overall MAE:  {mae:.3f}")
print(f"Overall RMSE: {rmse:.3f}")
print(f"Overall R²:   {r2:.3f}")
print(f"Overall Max Error: {max_err:.3f}")

# per-parking breakdown 
per_parking = (
    preds.assign(
        abs_err=(preds['spaces_left'] - preds['predicted']).abs(),
        sq_err =(preds['spaces_left'] - preds['predicted'])**2,
    )
    .groupby('parking_id')
    .agg(mae=('abs_err', 'mean'),
         rmse=('sq_err', lambda s: np.sqrt(s.mean())),
         n=('spaces_left', 'size'))
)
print(per_parking)

Overall MAE:  11.443
Overall RMSE: 21.503
Overall R²:   0.938
Overall Max Error: 228.152
                  mae       rmse      n
parking_id                             
2           10.701831  14.978214  73044
4           19.868988  30.991209  73189
5            7.049877  10.322478  73628
6           16.039697  31.123115  72683
7            3.566875   7.400622  72762


Bucketing model performs decently, especialy for how simple it is. MAE is worse by about 7 witch is significant but not huge. But Xgboost is isgnificantly better by all metrics, even overall max error is much better. This suggests that Xgboost is able to capture more complex patterns in the data, while bucketing model is limited by its simplicity.    

In [16]:
print(bucket_means.shape[0])

1680


1680 buckets - 2 types of weeks (acive classes, and holdiay breaks) * 7 days * 24 hours * 5 parkings = 1680